
# Manual Trading Challenge — Mean Field Simulation for Bid 2

This notebook simulates the **Celestial Gardeners' Guild** bidding problem using a mean-field / large-population game perspective.

## Goal
Each team wants to maximize its own expected PnL. There are roughly **4,100 teams**, and no team has an information advantage. Because the number of teams is large, one team's bid has almost no impact on the average second bid. This makes the game naturally close to a **mean field game**:

$$
\text{Instead of modeling every other team individually, model the population average / distribution of } b_2.
$$

## Rules encoded here
Reserve price grid:

$$
r \in \{670,675,680,\dots,920\}
$$

You submit two bids: $b_1$ and $b_2$.

- First bid fills iff $b_1 > r$.
- Second bid applies only when the first bid does not fill, and fills iff $b_2 > r$.
- If your second bid is strictly higher than the average second bid of all teams, you receive full PnL.
- If your second bid is less than or equal to the average second bid, the second-bid PnL is penalized by:

$$
\left(\frac{920 - \overline{b_2}}{920 - b_2}\right)^3
$$

The notebook helps answer:

1. What is a reasonable $b_1$?
2. What is a reasonable $b_2$ if everyone is strategically competing around the average?
3. Should we submit one fixed $b_2$, or randomize?
4. What graphs / numbers should we inspect before choosing final bids?


## Important Note 
#### by YZ
Section 1-7 assumes `b1=795` as it got solved earlier in the problem set, but since the dynamic among bid 1, bid 2, and mean of all bid 2 are related, section 7-end is joint mean-field over (b1, b2) of all possible (b1, b2).


In [34]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import Markdown, display

# If Plotly is missing in your kernel, run once in a notebook cell:
# %pip install plotly

FAIR_VALUE = 920
LOW_RESERVE = 670
HIGH_RESERVE = 920
STEP = 5
N_TEAMS = 4100        # Population size used to simulate the mean second bid.
TRADABLE = 500        # Starting assumption for how many counterparties we can trade.
FINAL_PNL_SIMS = 2000 # Monte Carlo paths for final PnL over TRADABLE independent reserves.

reserve_grid = np.arange(LOW_RESERVE, HIGH_RESERVE + STEP, STEP)
bid_grid = np.arange(LOW_RESERVE, HIGH_RESERVE + STEP, STEP)

PLOT_TEMPLATE = "plotly_white"


def clean_fig(fig, title, x_title, y_title, *, hovermode="x unified", height=500):
    """Apply consistent styling to interactive Plotly figures."""
    fig.update_layout(
        template=PLOT_TEMPLATE,
        title=title,
        xaxis_title=x_title,
        yaxis_title=y_title,
        hovermode=hovermode,
        height=height,
        legend_title_text="Click legend items to toggle",
        margin=dict(l=50, r=30, t=70, b=50),
    )
    return fig


def show_takeaway(text):
    """Render a short decision note below a chart."""
    display(Markdown(f"**What to look for:** {text}"))


print(f"Number of reserve points: {len(reserve_grid)}")
print(reserve_grid[:5], "...", reserve_grid[-5:])


Number of reserve points: 51
[670 675 680 685 690] ... [900 905 910 915 920]



## 1. Payoff logic

For reserve $r$:

- If $b_1 > r$, you buy at $b_1$ and earn $920-b_1$.
- If $b_1 \le r$ and $b_2 > r$, you buy at $b_2$ and earn $920-b_2$, possibly multiplied by the penalty factor.
- If $b_2 \le r$, no trade.

Because the inequality is **strict**, a bid equal to the reserve price does **not** fill.


In [35]:

def penalty_factor(b2, avg_b2, fair_value=FAIR_VALUE):
    """Penalty factor for second-bid PnL."""
    if b2 > avg_b2:
        return 1.0
    denom = fair_value - b2
    if denom <= 0:
        return 0.0
    return ((fair_value - avg_b2) / denom) ** 3


def expected_pnl_components(b1, b2, avg_b2, reserve_grid=reserve_grid):
    """Expected first, second, and total PnL per counterparty over the discrete reserve grid."""
    pen = penalty_factor(b2, avg_b2)
    first, second = [], []
    for r in reserve_grid:
        if b1 > r:
            first.append(FAIR_VALUE - b1); second.append(0.0)
        elif b2 > r:
            first.append(0.0); second.append((FAIR_VALUE - b2) * pen)
        else:
            first.append(0.0); second.append(0.0)
    return np.mean(first), np.mean(second), np.mean(first) + np.mean(second)


def expected_pnl_per_counterparty(b1, b2, avg_b2):
    return expected_pnl_components(b1, b2, avg_b2)[2]


def transaction_pnl(b1, b2, avg_b2, reserves):
    """PnL for each realized reserve draw."""
    reserves = np.asarray(reserves)
    pen = penalty_factor(b2, avg_b2)
    pnl = np.zeros_like(reserves, dtype=float)
    first_mask = b1 > reserves
    second_mask = (~first_mask) & (b2 > reserves)
    pnl[first_mask] = FAIR_VALUE - b1
    pnl[second_mask] = (FAIR_VALUE - b2) * pen
    return pnl


def simulate_final_pnl(
    b1,
    b2,
    avg_b2,
    tradable=TRADABLE,
    n_sims=FINAL_PNL_SIMS,
    seed=42,
):
    """Simulate final PnL as the sum of TRADABLE independent reserve transactions."""
    rng = np.random.default_rng(seed)
    reserve_draws = rng.choice(reserve_grid, size=(n_sims, tradable), replace=True)
    return transaction_pnl(b1, b2, avg_b2, reserve_draws).sum(axis=1)


def summarize_final_pnl(b1, b2, avg_b2, tradable=TRADABLE, n_sims=FINAL_PNL_SIMS, seed=42):
    samples = simulate_final_pnl(b1, b2, avg_b2, tradable=tradable, n_sims=n_sims, seed=seed)
    return {
        "final_pnl_mean": samples.mean(),
        "final_pnl_std": samples.std(ddof=1),
        "final_pnl_5pct": np.quantile(samples, 0.05),
        "final_pnl_95pct": np.quantile(samples, 0.95),
    }

expected_pnl_components(b1=795, b2=890, avg_b2=885)


(np.float64(61.27450980392157),
 np.float64(11.176470588235293),
 np.float64(72.45098039215686))


## 2. First bid benchmark

The first bid has no mean-field competition. It is only a margin-versus-fill tradeoff.

Because trade requires $b_1 > r$, a bid of 795 fills reserves up to 790, not 795.


In [36]:
def first_bid_only_pnl(b1):
    return np.where(b1 > reserve_grid, FAIR_VALUE - b1, 0.0).mean()


first_bid_df = pd.DataFrame({
    "b1": bid_grid,
    "first_bid_expected_pnl": [first_bid_only_pnl(b) for b in bid_grid],
})
first_bid_final = first_bid_df.apply(
    lambda row: pd.Series(summarize_final_pnl(row["b1"], LOW_RESERVE, LOW_RESERVE, seed=7000 + int(row["b1"]))),
    axis=1,
)
first_bid_df = pd.concat([first_bid_df, first_bid_final.add_prefix("first_bid_")], axis=1)
best_first = first_bid_df.loc[first_bid_df["first_bid_expected_pnl"].idxmax()]
display(best_first)

fig = px.line(
    first_bid_df,
    x="b1",
    y="first_bid_expected_pnl",
    markers=True,
    hover_data={
        "b1": ":.0f",
        "first_bid_expected_pnl": ":.4f",
        "first_bid_final_pnl_mean": ":.2f",
        "first_bid_final_pnl_5pct": ":.2f",
        "first_bid_final_pnl_95pct": ":.2f",
    },
)
fig.add_vline(
    x=best_first["b1"],
    line_dash="dash",
    annotation_text=f"best first-only b1={int(best_first['b1'])}",
    annotation_position="top left",
)
clean_fig(fig, "First-Bid-Only Expected PnL", "First bid b1", "Expected PnL per counterparty")
fig.show()

show_takeaway(
    "The peak is only a first-bid-only benchmark. In the joint game, a slightly higher b1 may be better "
    "because it avoids low-quality second-bid states and changes the population's b2 distribution."
)


b1                             795.000000
first_bid_expected_pnl          61.274510
first_bid_final_pnl_mean     30695.125000
first_bid_final_pnl_std       1401.556827
first_bid_final_pnl_5pct     28375.000000
first_bid_final_pnl_95pct    33000.000000
Name: 25, dtype: float64

**What to look for:** The peak is only a first-bid-only benchmark. In the joint game, a slightly higher b1 may be better because it avoids low-quality second-bid states and changes the population's b2 distribution.


### What to look for

- The best standalone first bid should be near the midpoint of the reserve range.
- Because the reserve grid is discrete and the rule is strict, use the exact grid result rather than only the continuous midpoint formula.
- In the full two-bid problem, **we may move $b_1$ slightly if doing so improves the second-bid opportunity or avoid second-bid altogether**.



## 3. Second bid as a best response to the mean field

Now treat the average second bid, $\overline{b_2}$, as fixed.

This is the core mean-field step:

$$
\text{Given population mean } \mu, \text{ what is my best response } b_2^*(\mu)?
$$

With 4,100 teams, your own bid has tiny impact on the average, so treating $\mu$ as exogenous is a good approximation.


In [37]:
def best_response_b2_given_mean(b1, avg_b2, b2_grid=None):
    if b2_grid is None:
        b2_grid = np.arange(b1 + STEP, FAIR_VALUE + STEP, STEP)
    rows = []
    for b2 in b2_grid:
        first, second, total = expected_pnl_components(b1, b2, avg_b2)
        rows.append({
            "b1": b1,
            "b2": b2,
            "avg_b2": avg_b2,
            "first_pnl": first,
            "second_pnl": second,
            "total_pnl": total,
            "penalty_factor": penalty_factor(b2, avg_b2),
            "beats_mean": b2 > avg_b2,
        })
    df = pd.DataFrame(rows)
    return df.sort_values("total_pnl", ascending=False), df


# Benchmark only: section 7 replaces this with a joint (b1,b2) mean-field model.
b1_base = int(best_first["b1"])
means_to_test = np.arange(850, 911, 5)
br_rows = []
for mu in means_to_test:
    ranked, _ = best_response_b2_given_mean(b1_base, mu)
    br_rows.append(ranked.iloc[0])

br_summary = pd.DataFrame(br_rows)[["avg_b2", "b2", "total_pnl", "second_pnl", "penalty_factor", "beats_mean"]]
br_final = br_summary.apply(
    lambda row: pd.Series(summarize_final_pnl(b1_base, row["b2"], row["avg_b2"], seed=1000 + int(row["avg_b2"]))),
    axis=1,
)
br_summary = pd.concat([br_summary, br_final], axis=1)
display(br_summary)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=br_summary["avg_b2"],
    y=br_summary["b2"],
    mode="lines+markers",
    name=f"best response b2 | b1={b1_base}",
    customdata=br_summary[["total_pnl", "final_pnl_mean", "final_pnl_5pct", "final_pnl_95pct", "second_pnl", "penalty_factor", "beats_mean"]],
    hovertemplate=(
        "avg_b2=%{x:.0f}<br>best b2=%{y:.0f}<br>"
        "total_pnl_per_counterparty=%{customdata[0]:.4f}<br>"
        "final_pnl_mean=%{customdata[1]:.2f}<br>"
        "final_pnl_5pct=%{customdata[2]:.2f}<br>final_pnl_95pct=%{customdata[3]:.2f}<br>"
        "second_pnl=%{customdata[4]:.4f}<br>"
        "penalty=%{customdata[5]:.4f}<br>beats_mean=%{customdata[6]}<extra></extra>"
    ),
))
fig.add_trace(go.Scatter(
    x=br_summary["avg_b2"],
    y=br_summary["avg_b2"],
    mode="lines",
    line=dict(dash="dash"),
    name="b2 = avg_b2",
    hovertemplate="avg_b2=%{x:.0f}<br>b2=%{y:.0f}<extra></extra>",
))
clean_fig(fig, f"Best-Response Second Bid Given Mean Field (benchmark b1={b1_base})", "Assumed population average second bid", "Best-response b2")
fig.show()

show_takeaway(
    "This graph still holds b1 fixed at the first-only optimum. If the best-response line sits above the diagonal, "
    "players are incentivized to outbid the mean; if it sits on/below the diagonal, penalty and margin trade off sharply."
)


,avg_b2,b2,total_pnl,second_pnl,penalty_factor,beats_mean,final_pnl_mean,final_pnl_std,final_pnl_5pct,final_pnl_95pct
12,850,860,76.568627,15.294118,1.0,True,38265.4450,1166.139043,36325.00,40195.50
12,855,860,76.568627,15.294118,1.0,True,38268.1000,1164.340246,36419.00,40195.00
12,860,860,76.568627,15.294118,1.0,False,38262.8600,1151.491310,36389.75,40155.25
13,865,865,76.372549,15.098039,1.0,False,38149.1400,1138.157321,36259.75,40055.00
14,870,870,75.980392,14.705882,1.0,False,38039.2250,1152.648341,36175.00,39975.00
15,875,875,75.392157,14.117647,1.0,False,37702.8125,1174.817986,35735.00,39610.00
16,880,880,74.607843,13.333333,1.0,False,37367.0325,1123.923411,35504.75,39205.00
17,885,885,73.627451,12.352941,1.0,False,36828.9300,1163.017642,34999.25,38770.50
18,890,890,72.450980,11.176471,1.0,False,36248.4500,1163.120468,34364.75,38190.25
19,895,895,71.078431,9.803922,1.0,False,35581.4625,1188.525545,33648.75,37551.25


**What to look for:** This graph still holds b1 fixed at the first-only optimum. If the best-response line sits above the diagonal, players are incentivized to outbid the mean; if it sits on/below the diagonal, penalty and margin trade off sharply.


### What to look for

This graph is very important.

- If the best-response line is close to the dashed line, the strategic logic is: **bid just above the expected average**.
- If the best response is below the dashed line, the model thinks the penalty is acceptable and margin dominates.
- If the best response jumps upward, the strict $>$ rule is forcing aggressive bidding.

In a stable large-population game, the population average should not be wildly inconsistent with the typical best response.



## 4. Payoff curves under different average-bid assumptions

Here we plot expected PnL as a function of your $b_2$ under several assumed population averages. This shows whether the optimum is sharp or flat.


In [38]:
selected_means = [860, 875, 890, 900]
fig = go.Figure()

for mu in selected_means:
    _, df = best_response_b2_given_mean(b1_base, mu)
    final_df = df.apply(
        lambda row: pd.Series(summarize_final_pnl(b1_base, row["b2"], mu, seed=2000 + int(mu) + int(row["b2"]))),
        axis=1,
    )
    df = pd.concat([df, final_df], axis=1)
    fig.add_trace(go.Scatter(
        x=df["b2"],
        y=df["total_pnl"],
        mode="lines+markers",
        name=f"avg_b2={mu}",
        customdata=df[["total_pnl", "final_pnl_mean", "final_pnl_5pct", "final_pnl_95pct", "first_pnl", "second_pnl", "penalty_factor", "beats_mean"]],
        hovertemplate=(
            "b2=%{x:.0f}<br>total_pnl_per_counterparty=%{customdata[0]:.4f}<br>"
            "final_pnl_mean=%{customdata[1]:.2f}<br>"
            "final_pnl_5pct=%{customdata[2]:.2f}<br>final_pnl_95pct=%{customdata[3]:.2f}<br>"
            "first_pnl=%{customdata[4]:.4f}<br>second_pnl=%{customdata[5]:.4f}<br>"
            "penalty=%{customdata[6]:.4f}<br>beats_mean=%{customdata[7]}<extra></extra>"
        ),
    ))

clean_fig(fig, f"Expected Total PnL vs b2 Under Different Mean Fields (benchmark b1={b1_base})", "Your second bid b2", "Expected PnL per counterparty")
fig.show()

show_takeaway(
    "Use the legend to toggle different assumed means. A robust b2 should not depend on one narrow mean-field guess; "
    "large curve shifts indicate that endogenous mean_b2 matters and motivates the joint model below."
)


**What to look for:** Use the legend to toggle different assumed means. A robust b2 should not depend on one narrow mean-field guess; large curve shifts indicate that endogenous mean_b2 matters and motivates the joint model below.


### What to look for

For each curve:

1. Locate the maximum.
2. Check whether the maximum is **just above the assumed mean**.
3. Check how costly it is to miss by 5 or 10 points.
4. If the curve drops sharply below the mean, avoiding the penalty is crucial.
5. If the curve drops sharply above the mean, overbidding destroys margin.

Decision rule:

$$
\text{Choose a } b_2 \text{ high enough to beat the expected mean, but not much higher.}
$$



## 5. Finite-population simulation with 4,100 teams

Now simulate many teams drawing $b_2$ from a proposed population strategy distribution.

For each simulated round:

1. Other teams draw their second bids.
2. We calculate the realized average second bid.
3. We evaluate our candidate bid against that realized average.
4. We repeat many times.

With 4,100 teams, the realized average should be quite stable unless the population strategy is very dispersed.


In [39]:
def sample_from_distribution(values, probs, size, rng):
    values = np.asarray(values)
    probs = np.asarray(probs, dtype=float)
    probs = probs / probs.sum()
    return rng.choice(values, size=size, p=probs)


def simulate_candidate_against_population(
    b1,
    my_b2,
    pop_values,
    pop_probs,
    n_teams=N_TEAMS,
    n_sims=20_000,
    seed=42,
    include_my_bid_in_average=True,
):
    rng = np.random.default_rng(seed)
    avg_samples = np.empty(n_sims)
    pnl_samples = np.empty(n_sims)
    final_pnl_samples = np.empty(n_sims)
    penalty_samples = np.empty(n_sims)
    beats_samples = np.empty(n_sims, dtype=bool)
    other_n = n_teams - 1 if include_my_bid_in_average else n_teams
    for i in range(n_sims):
        other_bids = sample_from_distribution(pop_values, pop_probs, other_n, rng)
        if include_my_bid_in_average:
            avg_b2 = (other_bids.sum() + my_b2) / n_teams
        else:
            avg_b2 = other_bids.mean()
        avg_samples[i] = avg_b2
        penalty_samples[i] = penalty_factor(my_b2, avg_b2)
        beats_samples[i] = my_b2 > avg_b2
        pnl_samples[i] = expected_pnl_per_counterparty(b1, my_b2, avg_b2)
        reserves = rng.choice(reserve_grid, size=TRADABLE, replace=True)
        final_pnl_samples[i] = transaction_pnl(b1, my_b2, avg_b2, reserves).sum()
    return pd.DataFrame({
        "avg_b2": avg_samples,
        "my_b2": my_b2,
        "beats_avg": beats_samples,
        "penalty_factor": penalty_samples,
        "pnl": pnl_samples,
        "final_pnl": final_pnl_samples,
    })


# Benchmark population distribution: centered around 885 with some dispersion.
pop_values = np.array([870, 875, 880, 885, 890, 895, 900])
pop_probs = np.array([0.05, 0.10, 0.20, 0.25, 0.20, 0.15, 0.05])

candidate_b2s = np.arange(865, 911, 5)
summary_rows = []
for my_b2 in candidate_b2s:
    sim = simulate_candidate_against_population(
        b1_base, my_b2, pop_values, pop_probs, n_sims=5000, seed=100 + int(my_b2)
    )
    mean_pnl = sim["pnl"].mean()
    pnl_5pct = sim["pnl"].quantile(0.05)
    pnl_95pct = sim["pnl"].quantile(0.95)
    final_pnl_mean = sim["final_pnl"].mean()
    final_pnl_5pct = sim["final_pnl"].quantile(0.05)
    final_pnl_95pct = sim["final_pnl"].quantile(0.95)
    summary_rows.append({
        "my_b2": my_b2,
        "mean_realized_avg": sim["avg_b2"].mean(),
        "std_realized_avg": sim["avg_b2"].std(),
        "prob_beat_avg": sim["beats_avg"].mean(),
        "mean_penalty": sim["penalty_factor"].mean(),
        "mean_pnl": mean_pnl,
        "pnl_5pct": pnl_5pct,
        "pnl_95pct": pnl_95pct,
        "final_pnl_mean": final_pnl_mean,
        "final_pnl_5pct": final_pnl_5pct,
        "final_pnl_95pct": final_pnl_95pct,
    })

candidate_summary = pd.DataFrame(summary_rows).sort_values("mean_pnl", ascending=False)
display(candidate_summary)

plot_df = candidate_summary.sort_values("my_b2")
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=plot_df["my_b2"],
    y=plot_df["pnl_95pct"],
    mode="lines",
    line=dict(width=0),
    showlegend=False,
    hoverinfo="skip",
))
fig.add_trace(go.Scatter(
    x=plot_df["my_b2"],
    y=plot_df["pnl_5pct"],
    mode="lines",
    fill="tonexty",
    fillcolor="rgba(31, 119, 180, 0.18)",
    line=dict(width=0),
    name="5%-95% PnL band",
    hovertemplate="b2=%{x:.0f}<br>5pct_pnl=%{y:.4f}<extra></extra>",
))
fig.add_trace(go.Scatter(
    x=plot_df["my_b2"],
    y=plot_df["mean_pnl"],
    mode="lines+markers",
    name="mean PnL per counterparty",
    customdata=plot_df[[
        "final_pnl_mean",
        "final_pnl_5pct",
        "final_pnl_95pct",
        "mean_realized_avg",
        "std_realized_avg",
        "pnl_5pct",
        "pnl_95pct",
    ]],
    hovertemplate=(
        "my_b2=%{x:.0f}<br>mean_pnl_per_counterparty=%{y:.4f}<br>"
        "final_pnl_mean=%{customdata[0]:.2f}<br>"
        "final_pnl_5pct=%{customdata[1]:.2f}<br>final_pnl_95pct=%{customdata[2]:.2f}<br>"
        "mean_realized_avg=%{customdata[3]:.4f}<br>std_realized_avg=%{customdata[4]:.4f}<br>"
        "pnl_5pct=%{customdata[5]:.4f}<br>pnl_95pct=%{customdata[6]:.4f}<extra></extra>"
    ),
))
clean_fig(fig, "Candidate b2 Performance Against Simulated Population (benchmark b1=795)", "Your second bid b2", "Expected PnL per counterparty")
fig.show()

fig = px.line(
    plot_df,
    x="my_b2",
    y="prob_beat_avg",
    markers=True,
    hover_data={
        "my_b2": ":.0f",
        "prob_beat_avg": ":.4f",
        "mean_penalty": ":.4f",
        "mean_pnl": ":.4f",
        "final_pnl_mean": ":.2f",
        "final_pnl_5pct": ":.2f",
        "final_pnl_95pct": ":.2f",
    },
)
clean_fig(fig, "Probability Your b2 Beats the Realized Average", "Your second bid b2", "Probability of beating average")
fig.show()

show_takeaway(
    "The band shows finite-population risk around the same b1=795 benchmark. The joint model below replaces this "
    "hand-chosen population distribution with an endogenous distribution over (b1,b2) pairs."
)


,my_b2,mean_realized_avg,std_realized_avg,prob_beat_avg,mean_penalty,mean_pnl,pnl_5pct,pnl_95pct,final_pnl_mean,final_pnl_5pct,final_pnl_95pct
4,885,885.499554,0.116792,0.0,0.957822,73.106432,72.910923,73.313788,36553.322456,34637.146139,38443.918318
5,890,885.500033,0.118595,1.0,1.000000,72.450980,72.450980,72.450980,36223.546000,34345.000000,38180.000000
6,895,885.500747,0.119148,1.0,1.000000,71.078431,71.078431,71.078431,35548.375000,33600.000000,37475.000000
3,880,885.498245,0.119288,0.0,0.641740,69.831044,69.688687,69.978159,34894.374118,32895.248070,36932.377520
7,900,885.503554,0.117331,1.0,1.000000,69.509804,69.509804,69.509804,34746.923000,32699.750000,36770.250000
8,905,885.501005,0.118749,1.0,1.000000,67.745098,67.745098,67.745098,33872.787000,31779.750000,35905.000000
2,875,885.497484,0.119624,0.0,0.450744,67.637961,67.527673,67.745576,33806.494358,31682.902747,35861.390473
1,870,885.494554,0.118498,0.0,0.328676,66.107984,66.026552,66.187957,33058.033310,30888.851310,35190.425284
9,910,885.506739,0.119926,1.0,1.000000,65.784314,65.784314,65.784314,32876.342000,30660.000000,35045.250000
0,865,885.495778,0.118665,0.0,0.246913,65.002413,64.938023,65.066496,32511.743305,30326.071236,34658.429955


**What to look for:** The band shows finite-population risk around the same b1=795 benchmark. The joint model below replaces this hand-chosen population distribution with an endogenous distribution over (b1,b2) pairs.


### What to look for

Important columns:

- `mean_pnl`: main objective. Higher is better.
- `prob_beat_avg`: probability you avoid the penalty.
- `mean_penalty`: average penalty factor when competition hurts you.
- `pnl_5pct`: downside case. Useful for robustness.
- `std_realized_avg`: uncertainty of the average. With 4,100 teams, this should usually be small.

Decision logic:

1. Start with the bid that has the highest `mean_pnl`.
2. If two bids are close, prefer better downside `pnl_5pct`.
3. Avoid bids with low `prob_beat_avg`, unless the margin advantage offsets the penalty.
4. Avoid bids that beat the average almost always but have low PnL, because that means you are overpaying.



## 6. Mean-field equilibrium approximation using multiplicative weights

We build a simple iterative algorithm:

1. Start with an initial population distribution over possible $b_2$ values.
2. Compute the current mean $\mu$.
3. Compute payoff for each possible $b_2$ against that mean.
4. Increase probability weight on higher-payoff bids.
5. Repeat.

This is not a proof of equilibrium, but it is a practical approximation of where strategic bidding may concentrate.


In [40]:
def multiplicative_weights_mean_field(b1, b2_values=None, n_iter=500, eta=0.05, temperature=1.0):
    """Legacy benchmark: solve only the second-bid mean field for one fixed b1."""
    if b2_values is None:
        b2_values = np.arange(b1 + STEP, FAIR_VALUE + STEP, STEP)
    b2_values = np.asarray(b2_values)
    probs = np.ones(len(b2_values)) / len(b2_values)
    history = []
    for t in range(n_iter):
        mu = np.dot(probs, b2_values)
        payoffs = np.array([expected_pnl_per_counterparty(b1, b2, mu) for b2 in b2_values])
        centered = payoffs - payoffs.max()
        weights = probs * np.exp(eta * centered / max(temperature, 1e-8))
        probs = weights / weights.sum()
        history.append({
            "iter": t,
            "mean_b2": mu,
            "max_payoff": payoffs.max(),
            "avg_strategy_payoff": np.dot(probs, payoffs),
            "best_response_b2": b2_values[payoffs.argmax()],
        })
    return pd.DataFrame(history), pd.DataFrame({"b2": b2_values, "prob": probs})


hist, eq_dist = multiplicative_weights_mean_field(b1_base, n_iter=1000, eta=0.15, temperature=1.0)
display(hist.tail())
display(eq_dist.sort_values("prob", ascending=False).head(10))

fig = px.line(
    hist,
    x="iter",
    y="mean_b2",
    hover_data={"iter": ":.0f", "mean_b2": ":.4f", "max_payoff": ":.4f", "best_response_b2": ":.0f"},
)
clean_fig(fig, "Benchmark Mean-Field Iteration: Population Average b2 (fixed b1=795)", "Iteration", "Mean b2")
fig.show()

fig = px.bar(
    eq_dist,
    x="b2",
    y="prob",
    hover_data={"b2": ":.0f", "prob": ":.6f"},
)
clean_fig(fig, f"Benchmark Mixed Strategy Distribution over b2 (fixed b1={b1_base})", "b2", "Probability weight")
fig.show()

show_takeaway(
    "This is useful only as a baseline. Because b1 changes how often b2 matters, section 7 lets the population learn "
    "over full (b1,b2) pairs instead of keeping b1 fixed."
)


,iter,mean_b2,max_payoff,avg_strategy_payoff,best_response_b2
995,995,880.0,74.607843,74.607843,880
996,996,880.0,74.607843,74.607843,880
997,997,880.0,74.607843,74.607843,880
998,998,880.0,74.607843,74.607843,880
999,999,880.0,74.607843,74.607843,880


,b2,prob
16,880,1.000000e+00
17,885,8.308827e-64
18,890,1.902554e-140
15,875,1.374945e-221
19,895,7.341203e-230
0,800,0.000000e+00
13,865,0.000000e+00
23,915,0.000000e+00
22,910,0.000000e+00
21,905,0.000000e+00


**What to look for:** This is useful only as a baseline. Because b1 changes how often b2 matters, section 7 lets the population learn over full (b1,b2) pairs instead of keeping b1 fixed.


### What to look for

From the mean-field iteration graph:

- Does the mean stabilize?
- Does the best response remain near the mean plus one tick?
- Does the distribution concentrate in a narrow band or remain dispersed?

From the distribution plot:

- The highest-probability region is where strategic teams may cluster.
- If the distribution clusters around 885-900, then choosing a fixed $b_2$ slightly above that cluster may be attractive.
- But if everyone thinks this way, the cluster moves upward. That is why the fixed-point logic matters.

Important warning: this algorithm approximates the **population's strategic center**, not necessarily your final bid. Your final bid should be a best response to the expected realized population, not blindly equal to the population mean.


## 7. Joint mean-field over $(b_1,b_2)$ pairs

The earlier sections intentionally held $b_1=795$ fixed as a benchmark. That is not the full game.

In the actual challenge, $b_1$ and $b_2$ interact:

- Raising $b_1$ increases first-bid fills and may avoid the second-bid scenario entirely.
- Avoiding more second-bid scenarios changes how valuable each $b_2$ is.
- If many teams change $(b_1,b_2)$ together, the population mean $\overline{b_2}$ also changes.

So this section models the population strategy as a probability distribution over **legal pairs**:

$$
(b_1,b_2) \quad \text{with} \quad b_2 > b_1
$$

At each iteration:

1. Compute the population mean $\mu_t = E[b_2]$ from the current pair distribution.
2. Score every candidate pair $(b_1,b_2)$ against that same $\mu_t$.
3. Update the pair distribution toward higher-payoff pairs using multiplicative weights.

This is the cleaner joint mean-field approximation: $b_1$, $b_2$, and $\overline{b_2}$ are solved together rather than assumed separately.


In [41]:
def make_pair_grid(b1_values=None, b2_values=None):
    """Create all legal (b1,b2) pairs with b2 > b1."""
    if b1_values is None:
        b1_values = np.arange(760, 831, STEP)
    if b2_values is None:
        b2_values = np.arange(830, 921, STEP)

    pairs = [
        {"b1": int(b1), "b2": int(b2)}
        for b1 in b1_values
        for b2 in b2_values
        if b2 > b1
    ]
    return pd.DataFrame(pairs)


def score_pairs_against_mean(pair_grid, avg_b2):
    """Score every pair against one population mean b2."""
    rows = []
    for row in pair_grid.itertuples(index=False):
        first, second, total = expected_pnl_components(row.b1, row.b2, avg_b2)
        rows.append({
            "b1": row.b1,
            "b2": row.b2,
            "avg_b2": avg_b2,
            "first_pnl": first,
            "second_pnl": second,
            "total_pnl": total,
            "expected_final_pnl_mean": total * TRADABLE,
            "beats_mean": row.b2 > avg_b2,
            "penalty_factor": penalty_factor(row.b2, avg_b2),
        })
    return pd.DataFrame(rows)


def joint_mean_field_pairs(
    b1_values=None,
    b2_values=None,
    n_iter=2000,
    eta=0.12,
    temperature=1.0,
    inertia=0.05,
):
    """Multiplicative-weights mean field over full (b1,b2) pairs.

    `inertia` keeps a small amount of exploration mass on every legal pair so the
    distribution does not collapse too early onto one noisy local optimum.

    eta: rate of learning; let population learn faster to achieve better pnls
    temperature: high temp -> more exploration; low temp -> more exploitation
    inertia: keep some exploration to avoid local optima


    """
    pair_grid = make_pair_grid(b1_values, b2_values)
    probs = np.ones(len(pair_grid), dtype=float) / len(pair_grid)
    history = []

    for t in range(n_iter):
        mu = float(np.dot(probs, pair_grid["b2"]))
        scored = score_pairs_against_mean(pair_grid, mu)
        payoffs = scored["total_pnl"].to_numpy()
        centered = payoffs - payoffs.max()

        weights = probs * np.exp(eta * centered / max(temperature, 1e-8))
        probs = weights / weights.sum()
        if inertia > 0:
            probs = (1 - inertia) * probs + inertia / len(probs)
            probs = probs / probs.sum()

        best_idx = int(payoffs.argmax())
        history.append({
            "iter": t,
            "mean_b2": mu,
            "mean_b1": float(np.dot(probs, pair_grid["b1"])),
            "avg_strategy_payoff": float(np.dot(probs, payoffs)),
            "max_payoff": float(payoffs.max()),
            "best_response_b1": int(pair_grid.iloc[best_idx]["b1"]),
            "best_response_b2": int(pair_grid.iloc[best_idx]["b2"]),
            "best_response_gap": float(payoffs.max() - np.dot(probs, payoffs)),
        })

    dist = pair_grid.copy()
    dist["prob"] = probs
    final_mu = float(np.dot(probs, dist["b2"]))
    final_scores = score_pairs_against_mean(pair_grid, final_mu)
    dist = dist.merge(final_scores, on=["b1", "b2"], how="left")
    return pd.DataFrame(history), dist.sort_values("prob", ascending=False), final_scores


joint_hist, joint_dist, joint_scores = joint_mean_field_pairs(
    b1_values=np.arange(760, 831, STEP),
    b2_values=np.arange(830, 921, STEP),
    n_iter=2000,
    eta=0.12,
    temperature=1.0,
    inertia=0.03,
)

joint_mu = float(joint_hist["mean_b2"].tail(100).mean())
joint_mu_tick = round(joint_mu / STEP) * STEP

print(f"Joint equilibrium mean_b2 ≈ {joint_mu:.3f} (rounded tick: {joint_mu_tick})")
print(f"Joint equilibrium mean_b1 ≈ {joint_hist['mean_b1'].tail(100).mean():.3f}")
display(joint_hist.tail())
display(joint_dist.head(20))

# Convergence diagnostics.
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=joint_hist["iter"], y=joint_hist["mean_b2"],
    mode="lines", name="mean_b2",
    hovertemplate="iter=%{x}<br>mean_b2=%{y:.4f}<extra></extra>",
))
fig.add_trace(go.Scatter(
    x=joint_hist["iter"], y=joint_hist["mean_b1"],
    mode="lines", name="mean_b1",
    hovertemplate="iter=%{x}<br>mean_b1=%{y:.4f}<extra></extra>",
))
fig.add_trace(go.Scatter(
    x=joint_hist["iter"], y=joint_hist["best_response_b2"],
    mode="lines", name="best-response b2",
    line=dict(dash="dash"),
    hovertemplate="iter=%{x}<br>best_response_b2=%{y:.0f}<extra></extra>",
))
clean_fig(fig, "Joint Mean-Field Convergence over (b1,b2) Pairs", "Iteration", "Bid level")
fig.show()

show_takeaway(
    "Convergence is good when mean_b1 / mean_b2 stop drifting and the best-response gap stabilizes near zero. "
    "If mean_b1 is materially above 795, the joint game says it is worth paying more upfront to reduce bad second-stage exposure."
)

# Heatmap of final probability mass over pairs.
prob_pivot = joint_dist.pivot(index="b1", columns="b2", values="prob").fillna(0).sort_index()
fig = go.Figure(data=go.Heatmap(
    z=prob_pivot.values,
    x=prob_pivot.columns,
    y=prob_pivot.index,
    colorscale="Viridis",
    hovertemplate="b1=%{y}<br>b2=%{x}<br>prob=%{z:.6f}<extra></extra>",
))
clean_fig(fig, "Joint Mean-Field Final Distribution over (b1,b2)", "b2", "b1", hovermode="closest", height=650)
fig.show()

# Heatmap of final expected PnL at the endogenous joint mean.
score_pivot = joint_scores.pivot(index="b1", columns="b2", values="total_pnl").sort_index()
fig = go.Figure(data=go.Heatmap(
    z=score_pivot.values,
    x=score_pivot.columns,
    y=score_pivot.index,
    colorscale="RdYlGn",
    hovertemplate="b1=%{y}<br>b2=%{x}<br>PnL=%{z:.4f}<extra></extra>",
))
clean_fig(fig, f"Expected PnL Surface at Endogenous mean_b2≈{joint_mu:.2f}", "b2", "b1", hovermode="closest", height=650)
fig.show()

# Marginals make it easier to see which b1/b2 levels the population concentrates on.
b1_marginal = joint_dist.groupby("b1", as_index=False)["prob"].sum()
b2_marginal = joint_dist.groupby("b2", as_index=False)["prob"].sum()

fig = go.Figure()
fig.add_trace(go.Bar(
    x=b1_marginal["b1"], y=b1_marginal["prob"], name="b1 marginal",
    hovertemplate="b1=%{x}<br>prob=%{y:.6f}<extra></extra>",
))
fig.add_trace(go.Bar(
    x=b2_marginal["b2"], y=b2_marginal["prob"], name="b2 marginal",
    hovertemplate="b2=%{x}<br>prob=%{y:.6f}<extra></extra>",
))
clean_fig(fig, "Joint Mean-Field Marginal Bid Distributions", "Bid level", "Probability mass")
fig.show()


Joint equilibrium mean_b2 ≈ 884.675 (rounded tick: 885)
Joint equilibrium mean_b1 ≈ 778.246


,iter,mean_b2,mean_b1,avg_strategy_payoff,max_payoff,best_response_b1,best_response_b2,best_response_gap
1995,1995,884.674688,778.246443,74.361516,74.803922,775,885,0.442405
1996,1996,884.674688,778.246443,74.361516,74.803922,775,885,0.442405
1997,1997,884.674688,778.246443,74.361516,74.803922,775,885,0.442405
1998,1998,884.674688,778.246443,74.361516,74.803922,775,885,0.442405
1999,1999,884.674688,778.246443,74.361516,74.803922,775,885,0.442405


,b1,b2,prob,avg_b2,first_pnl,second_pnl,total_pnl,expected_final_pnl_mean,beats_mean,penalty_factor
87,780,885,0.469657,884.674688,60.392157,14.411765,74.803922,37401.960784,True,1.0
68,775,885,0.469657,884.674688,59.705882,15.098039,74.803922,37401.960784,True,1.0
49,770,885,0.004500,884.674688,58.823529,15.784314,74.607843,37303.921569,True,1.0
106,785,885,0.004500,884.674688,60.882353,13.725490,74.607843,37303.921569,True,1.0
125,790,885,0.001545,884.674688,61.176471,13.039216,74.215686,37107.843137,True,1.0
30,765,885,0.001545,884.674688,57.745098,16.470588,74.215686,37107.843137,True,1.0
11,760,885,0.000801,884.674688,56.470588,17.156863,73.627451,36813.725490,True,1.0
144,795,885,0.000801,884.674688,61.274510,12.352941,73.627451,36813.725490,True,1.0
88,780,890,0.000652,884.674688,60.392157,12.941176,73.333333,36666.666667,True,1.0
107,785,890,0.000615,884.674688,60.882353,12.352941,73.235294,36617.647059,True,1.0


**What to look for:** Convergence is good when mean_b1 / mean_b2 stop drifting and the best-response gap stabilizes near zero. If mean_b1 is materially above 795, the joint game says it is worth paying more upfront to reduce bad second-stage exposure.

### What to look for

Use the interactive charts above in this order:

1. **Convergence chart**: mean `b1`, mean `b2`, and best-response `b2` should settle. If they oscillate, rerun with lower `eta` or higher `inertia`.
2. **Final distribution heatmap**: do not just take the single highest-probability cell. Look for clusters of probability mass.
3. **PnL heatmap at endogenous `mean_b2`**: compare the brightest PnL region with the distribution heatmap. A good candidate is both high-PnL and close to where the population concentrates.
4. **Marginals**: if the `b1` marginal shifts above 795, that supports your intuition that paying more upfront can avoid low-quality second-bid situations.
5. **Hover values**: hover cells to read exact `(b1, b2, prob)` and `(b1, b2, PnL)` numbers before deciding.


## 8. Robustness around the joint equilibrium

The joint mean-field gives an endogenous estimate of the population mean second bid. But the real population may not converge exactly to that distribution.

So we stress-test promising pairs around the joint equilibrium:

- `mu_scenarios` is centered on the joint equilibrium `mean_b2`.
- Candidate pairs include the top-probability joint strategies and the top-PnL cells at the endogenous mean.
- We rank by average PnL, worst-case PnL, PnL volatility, and proximity to the joint distribution.

This keeps the interaction between $b_1$, $b_2$, and $\overline{b_2}$ while still protecting against model error.

### Note
Stress test by 4 steps (20 around `mu_center`)

robust score = worst_pnl_across_mu
    + 0.25 * avg_pnl_across_mu
    - 0.50 * std_pnl_across_mu


In [42]:
# Stress scenarios centered on the joint equilibrium mean_b2.
mu_center = joint_mu_tick
mu_scenarios = np.arange(mu_center - 20, mu_center + 21, STEP)

# Candidate set: top probability mass + top PnL at joint_mu + nearby first-only benchmark pairs.
top_prob_pairs = joint_dist.head(30)[["b1", "b2"]]
top_pnl_pairs = joint_scores.sort_values("total_pnl", ascending=False).head(30)[["b1", "b2"]]
benchmark_neighborhood = pd.DataFrame([
    {"b1": b1, "b2": b2}
    for b1 in np.arange(b1_base - 20, b1_base + 31, STEP)
    for b2 in np.arange(mu_center - 20, mu_center + 31, STEP)
    if b2 > b1
])

candidate_pairs = (
    pd.concat([top_prob_pairs, top_pnl_pairs, benchmark_neighborhood], ignore_index=True)
      .drop_duplicates()
      .sort_values(["b1", "b2"])
      .reset_index(drop=True)
)

robust_rows = []
for row in candidate_pairs.itertuples(index=False):
    pnls = np.array([expected_pnl_per_counterparty(row.b1, row.b2, mu) for mu in mu_scenarios])
    prob_mass = joint_dist.loc[(joint_dist["b1"] == row.b1) & (joint_dist["b2"] == row.b2), "prob"].sum()
    avg_pnl = pnls.mean()
    worst_pnl = pnls.min()
    std_pnl = pnls.std()
    best_pnl = pnls.max()
    center_final = summarize_final_pnl(
        row.b1,
        row.b2,
        mu_center,
        n_sims=1000,
        seed=3000 + int(row.b1) * 10 + int(row.b2),
    )
    robust_rows.append({
        "b1": row.b1,
        "b2": row.b2,
        "joint_prob": prob_mass,
        "avg_pnl_across_mu": avg_pnl,
        "avg_expected_final_pnl": avg_pnl * TRADABLE,
        "worst_pnl_across_mu": worst_pnl,
        "worst_expected_final_pnl": worst_pnl * TRADABLE,
        "std_pnl_across_mu": std_pnl,
        "std_expected_final_pnl": std_pnl * TRADABLE,
        "best_mu_case_pnl": best_pnl,
        "best_expected_final_pnl": best_pnl * TRADABLE,
        **center_final,
        "mu_min": mu_scenarios.min(),
        "mu_max": mu_scenarios.max(),
        "pair": f"({int(row.b1)}, {int(row.b2)})",
    })

robust_df = pd.DataFrame(robust_rows)
robust_df["robust_score"] = (
    robust_df["worst_pnl_across_mu"]
    + 0.25 * robust_df["avg_pnl_across_mu"]
    - 0.50 * robust_df["std_pnl_across_mu"]
)

ranked_robust = robust_df.sort_values(
    ["robust_score", "avg_pnl_across_mu", "joint_prob"],
    ascending=False,
)
display(ranked_robust.head(25))

top = ranked_robust.head(15).copy()

fig = px.bar(
    top,
    x="pair",
    y="avg_pnl_across_mu",
    color="joint_prob",
    hover_data={
        "b1": ":.0f",
        "b2": ":.0f",
        "joint_prob": ":.6f",
        "avg_pnl_across_mu": ":.4f",
        "avg_expected_final_pnl": ":.2f",
        "worst_pnl_across_mu": ":.4f",
        "worst_expected_final_pnl": ":.2f",
        "std_pnl_across_mu": ":.4f",
        "std_expected_final_pnl": ":.2f",
        "final_pnl_mean": ":.2f",
        "final_pnl_5pct": ":.2f",
        "final_pnl_95pct": ":.2f",
        "robust_score": ":.4f",
    },
)
clean_fig(fig, "Top Candidate Pairs by Average PnL (stress-tested around joint mean_b2)", "(b1, b2)", "Average PnL per counterparty across mu scenarios")
fig.show()

fig = px.bar(
    top,
    x="pair",
    y="worst_pnl_across_mu",
    color="joint_prob",
    hover_data={
        "b1": ":.0f",
        "b2": ":.0f",
        "joint_prob": ":.6f",
        "avg_pnl_across_mu": ":.4f",
        "avg_expected_final_pnl": ":.2f",
        "worst_pnl_across_mu": ":.4f",
        "worst_expected_final_pnl": ":.2f",
        "std_pnl_across_mu": ":.4f",
        "std_expected_final_pnl": ":.2f",
        "final_pnl_mean": ":.2f",
        "final_pnl_5pct": ":.2f",
        "final_pnl_95pct": ":.2f",
        "robust_score": ":.4f",
    },
)
clean_fig(fig, "Top Candidate Pairs by Worst-Case PnL (stress-tested around joint mean_b2)", "(b1, b2)", "Worst-case PnL per counterparty across mu scenarios")
fig.show()

# Toggleable line chart: each candidate pair's PnL curve across mu scenarios.
fig = go.Figure()
for row in top.itertuples(index=False):
    pnls = [expected_pnl_per_counterparty(row.b1, row.b2, mu) for mu in mu_scenarios]
    final_means = [
        summarize_final_pnl(row.b1, row.b2, mu, n_sims=1000, seed=4000 + int(row.b1) * 10 + int(row.b2) + int(mu))["final_pnl_mean"]
        for mu in mu_scenarios
    ]
    fig.add_trace(go.Scatter(
        x=mu_scenarios,
        y=pnls,
        mode="lines+markers",
        name=row.pair,
        customdata=np.column_stack([
            np.full(len(mu_scenarios), row.b1),
            np.full(len(mu_scenarios), row.b2),
            final_means,
            np.full(len(mu_scenarios), row.joint_prob),
        ]),
        hovertemplate=(
            "mu=%{x:.0f}<br>PnL_per_counterparty=%{y:.4f}<br>"
            "final_pnl_mean=%{customdata[2]:.2f}<br>"
            "b1=%{customdata[0]:.0f}<br>b2=%{customdata[1]:.0f}<br>"
            "joint_prob=%{customdata[3]:.6f}<extra></extra>"
        ),
    ))
clean_fig(fig, "Candidate Pair PnL Curves Across Mean-b2 Stress Scenarios", "Assumed mean_b2", "Expected PnL per counterparty")
fig.show()

show_takeaway(
    "Prefer pairs that are high in both average and worst-case PnL, with low std. If a pair has strong PnL but near-zero "
    "joint_prob, it may be an exploitive best response rather than a stable population-like choice."
)


,b1,b2,joint_prob,avg_pnl_across_mu,avg_expected_final_pnl,worst_pnl_across_mu,worst_expected_final_pnl,std_pnl_across_mu,std_expected_final_pnl,best_mu_case_pnl,best_expected_final_pnl,final_pnl_mean,final_pnl_std,final_pnl_5pct,final_pnl_95pct,mu_min,mu_max,pair,robust_score
37,785,905,0.000188,67.941176,33970.588235,67.941176,33970.588235,0.000000,0.000000,67.941176,33970.588235,33966.480,1388.540080,31799.25,36225.75,865,905,"(785, 905)",84.926471
48,790,905,0.000188,67.941176,33970.588235,67.941176,33970.588235,0.000000,0.000000,67.941176,33970.588235,33916.545,1262.683534,31864.25,35916.25,865,905,"(790, 905)",84.926471
26,780,905,0.000185,67.745098,33872.549020,67.745098,33872.549020,0.000000,0.000000,67.745098,33872.549020,33875.095,1408.741146,31503.50,36150.50,865,905,"(780, 905)",84.681373
59,795,905,0.000185,67.745098,33872.549020,67.745098,33872.549020,0.000000,0.000000,67.745098,33872.549020,33882.865,1277.032039,31774.25,35951.75,865,905,"(795, 905)",84.681373
70,800,905,0.000179,67.352941,33676.470588,67.352941,33676.470588,0.000000,0.000000,67.352941,33676.470588,33655.110,1206.892343,31679.25,35640.00,865,905,"(800, 905)",84.191176
15,775,905,0.000179,67.352941,33676.470588,67.352941,33676.470588,0.000000,0.000000,67.352941,33676.470588,33652.080,1443.866744,31309.50,36025.50,865,905,"(775, 905)",84.191176
81,805,905,0.000171,66.764706,33382.352941,66.764706,33382.352941,0.000000,0.000000,66.764706,33382.352941,33373.805,1121.664022,31569.75,35245.25,865,905,"(805, 905)",83.455882
92,810,905,0.000162,65.980392,32990.196078,65.980392,32990.196078,0.000000,0.000000,65.980392,32990.196078,32962.450,1115.070953,31150.00,34825.25,865,905,"(810, 905)",82.475490
49,790,910,0.000161,65.882353,32941.176471,65.882353,32941.176471,0.000000,0.000000,65.882353,32941.176471,32982.560,1405.889032,30710.00,35321.50,865,905,"(790, 910)",82.352941
38,785,910,0.000160,65.784314,32892.156863,65.784314,32892.156863,0.000000,0.000000,65.784314,32892.156863,32841.890,1364.274082,30584.75,35147.75,865,905,"(785, 910)",82.230392


**What to look for:** Prefer pairs that are high in both average and worst-case PnL, with low std. If a pair has strong PnL but near-zero joint_prob, it may be an exploitive best response rather than a stable population-like choice.

## 9. Final PnL Distribution by Candidate Pair

The robust-score table is useful for ranking, but final submission risk is easier to see as a distribution.

This section simulates full-round PnL for the top candidate `(b1, b2)` pairs. Each simulation path draws `TRADABLE` independent reserve prices from the discrete `670..920` grid, computes the PnL of each transaction, and sums them into one `final_pnl` sample.

Use the histogram to compare shape and overlap. Use the ECDF to compare downside: the curve that sits further right in the left tail is safer.

In [43]:
# Final PnL distribution for selected robust candidate pairs.
# Assumes section 8 has already produced `ranked_robust` and `joint_mu_tick`.
DIST_N_SIMS = 10_000
DIST_TOP_N = 6
RISK_AVERSION = 1.0

if "ranked_robust" not in globals():
    raise RuntimeError("Run section 8 first so `ranked_robust` is available.")

candidate_pairs_for_dist = (
    ranked_robust[["b1", "b2", "joint_prob", "robust_score"]]
    .head(DIST_TOP_N)
    .drop_duplicates(["b1", "b2"])
    .reset_index(drop=True)
)

dist_rows = []
summary_rows = []
for row in candidate_pairs_for_dist.itertuples(index=False):
    samples = simulate_final_pnl(
        b1=row.b1,
        b2=row.b2,
        avg_b2=joint_mu_tick,
        tradable=TRADABLE,
        n_sims=DIST_N_SIMS,
        seed=8000 + int(row.b1) * 10 + int(row.b2),
    )
    pair_label = f"({int(row.b1)}, {int(row.b2)})"
    summary_rows.append({
        "pair": pair_label,
        "b1": int(row.b1),
        "b2": int(row.b2),
        "joint_prob": row.joint_prob,
        "robust_score": row.robust_score,
        "final_pnl_mean": samples.mean(),
        "final_pnl_std": samples.std(ddof=1),
        "final_pnl_5pct": np.quantile(samples, 0.05),
        "final_pnl_50pct": np.quantile(samples, 0.50),
        "final_pnl_95pct": np.quantile(samples, 0.95),
    })
    dist_rows.extend({"pair": pair_label, "b1": int(row.b1), "b2": int(row.b2), "final_pnl": pnl} for pnl in samples)

dist_summary = pd.DataFrame(summary_rows)
dist_summary["risk_adjusted_score"] = dist_summary["final_pnl_mean"] - RISK_AVERSION * dist_summary["final_pnl_std"]
dist_summary = dist_summary.sort_values("risk_adjusted_score", ascending=False)
display(dist_summary)

dist_df = pd.DataFrame(dist_rows)

fig = px.histogram(
    dist_df,
    x="final_pnl",
    color="pair",
    nbins=80,
    barmode="overlay",
    opacity=0.45,
    marginal="box",
    histnorm="probability density",
)
clean_fig(
    fig,
    f"Final PnL Distribution over {TRADABLE} Independent Trades",
    "Final PnL",
    "Density",
)
fig.show()

fig = px.ecdf(
    dist_df,
    x="final_pnl",
    color="pair",
)
clean_fig(
    fig,
    "Final PnL ECDF by Candidate Pair",
    "Final PnL",
    "Cumulative probability",
)
fig.show()

show_takeaway(
    "Use `risk_adjusted_score = mean - λ·std` for a tunable risk penalty, and use the ECDF left tail to avoid pairs "
    "whose expected edge comes with poor downside. Increase `TRADABLE` to see distributions tighten; decrease it to stress luck risk."
)


,pair,b1,b2,joint_prob,robust_score,final_pnl_mean,final_pnl_std,final_pnl_5pct,final_pnl_50pct,final_pnl_95pct,risk_adjusted_score
1,"(790, 905)",790,905,0.000188,84.926471,33988.8775,1320.045823,31800.00,33980.0,36170.25,32668.831677
3,"(795, 905)",795,905,0.000185,84.681373,33892.2645,1261.279226,31829.75,33915.0,35945.00,32630.985274
0,"(785, 905)",785,905,0.000188,84.926471,33972.2970,1345.031154,31770.00,33975.0,36180.00,32627.265846
2,"(780, 905)",780,905,0.000185,84.681373,33885.6580,1405.630936,31590.00,33885.0,36210.00,32480.027064
4,"(800, 905)",800,905,0.000179,84.191176,33666.7665,1229.104469,31650.00,33660.0,35700.00,32437.662031
5,"(775, 905)",775,905,0.000179,84.191176,33681.0655,1455.677183,31310.00,33680.0,36065.25,32225.388317


**What to look for:** Use `risk_adjusted_score = mean - λ·std` for a tunable risk penalty, and use the ECDF left tail to avoid pairs whose expected edge comes with poor downside. Increase `TRADABLE` to see distributions tighten; decrease it to stress luck risk.

### What to look for

For final decision-making, compare three things together:

1. **Endogenous stability**: `joint_prob` from the joint mean-field distribution.
2. **Expected edge**: `avg_pnl_across_mu` over stress scenarios around the joint mean.
3. **Downside protection**: `worst_pnl_across_mu` and low `std_pnl_across_mu`.

A good final bid pair should usually live in one of two categories:

- **Stable population-like pair**: high `joint_prob`, decent robust PnL. This is safer if many teams reason similarly.
- **Exploitive best-response pair**: low `joint_prob` but high robust score. This may outperform if you believe the population mean is predictable and others cluster near the joint distribution.

Use the interactive PnL-curve plot to toggle candidate pairs and see which ones break when `mean_b2` moves by ±20 ticks.


## 9. Final decision framework

Use this process before submitting bids:

### Step 1 — Treat 795 as a benchmark, not an anchor

The first-only optimum is useful, but the joint model is allowed to move $b_1$ higher or lower.

Ask from section 7:

- Did the joint `mean_b1` move above 795?
- Does the `b1` marginal put meaningful probability above 795?
- Is the PnL heatmap bright at higher $b_1$ values?

If yes, the game is saying that avoiding bad second-bid exposure is worth some first-bid margin.

### Step 2 — Use the joint equilibrium to estimate population $b_2$

Use:

- `joint_mu`
- `joint_mu_tick`
- the `b2` marginal distribution
- the final distribution heatmap

This is better than assuming one fixed external $\mu$, because the mean is produced by the same pair distribution that determines $b_1$.

### Step 3 — Pick candidates from both stable and exploitive regions

Create a short candidate list from:

1. High `joint_prob` pairs in `joint_dist.head(20)`.
2. High-PnL cells in `joint_scores` at endogenous `joint_mu`.
3. High robust-score pairs in `ranked_robust.head(25)`.

Then compare them in the interactive stress chart.

### Step 4 — Check robustness, not just peak PnL

Do not choose a pair that only works at one exact population mean.

Prioritize:

- high `worst_pnl_across_mu`
- low `std_pnl_across_mu`
- stable PnL curve across `mu_scenarios`
- reasonable proximity to joint distribution mass

### Step 5 — Final selection rule

A practical final pair should satisfy most of these:

- $b_2 > b_1$ by enough ticks to preserve the second-bid option.
- $b_2$ is near or slightly above the joint `mean_b2` if you want to beat the mean.
- $b_1$ is not blindly fixed at 795; it is chosen based on the joint distribution and heatmap.
- The pair performs well across the ±20 tick stress range around `joint_mu_tick`.

## Main intuition

This is a **joint large-population game over bid pairs**, not a two-step problem where $b_1$ is solved first and $b_2$ is solved later.

The better mental model is:

$$
\text{The population chooses a distribution over }(b_1,b_2),\text{ which creates }\overline{b_2}.
$$

Then we choose whether to:

$$
\boxed{\text{play near the stable population cluster, or exploit it with a robust best response.}}
$$


## 10. Playground: manually stress-test b1 and TRADABLE

This section lets you manually choose a first bid `b1` and a tradable counterparty count, then see how the fixed-`b1` mean-field responds.

For a chosen `b1`, the playground:

1. Runs the fixed-`b1` multiplicative-weights mean-field over possible `b2` values.
2. Estimates the resulting population `mean_b2`.
3. Computes the best-response `b2` at that final rounded mean.
4. Shows the expected PnL split between first bid and second bid.
5. Simulates final PnL over the chosen number of independent tradable reserve draws.
6. Compares the manual `b1` result with the joint `(b1,b2)` equilibrium from section 7.

This is not meant to replace the joint analysis. It is a diagnostic tool: "If I insist on this `b1` and this `TRADABLE`, what happens to `b2`, the mean field, and the full final-PnL distribution?"

In [44]:
def fixed_b1_playground_result(
    b1,
    b2_values=None,
    n_iter=1000,
    eta=0.15,
    temperature=1.0,
    tradable=TRADABLE,
):
    """Run a fixed-b1 mean-field and summarize the resulting best-response b2.

    This answers: if we manually force the first bid to be `b1`, where does the
    population's second-bid mean settle, and what b2 should we choose against it?
    """
    b1 = int(b1)
    tradable = int(tradable)
    if b2_values is None:
        b2_values = np.arange(b1 + STEP, FAIR_VALUE + STEP, STEP)
    else:
        b2_values = np.asarray([b2 for b2 in b2_values if b2 > b1], dtype=int)

    if len(b2_values) == 0:
        raise ValueError(f"No legal b2 values remain for b1={b1}; require b2 > b1.")

    hist, dist = multiplicative_weights_mean_field(
        b1,
        b2_values=b2_values,
        n_iter=n_iter,
        eta=eta,
        temperature=temperature,
    )

    mean_b2 = float(hist["mean_b2"].tail(min(100, len(hist))).mean())
    rounded_mean_b2 = int(round(mean_b2 / STEP) * STEP)
    ranked, all_b2 = best_response_b2_given_mean(b1, rounded_mean_b2, b2_grid=b2_values)
    best = ranked.iloc[0].copy()

    joint_mean_b1 = float(joint_hist["mean_b1"].tail(100).mean()) if "joint_hist" in globals() else np.nan
    joint_mean_b2 = float(joint_mu) if "joint_mu" in globals() else np.nan
    best_final = summarize_final_pnl(
        b1,
        best["b2"],
        rounded_mean_b2,
        tradable=tradable,
        seed=5000 + int(b1) * 10 + int(best["b2"]),
    )

    summary = pd.DataFrame([{
        "chosen_b1": b1,
        "tradable": tradable,
        "mean_b2": mean_b2,
        "rounded_mean_b2": rounded_mean_b2,
        "best_response_b2": int(best["b2"]),
        "first_pnl": float(best["first_pnl"]),
        "second_pnl": float(best["second_pnl"]),
        "total_pnl": float(best["total_pnl"]),
        **best_final,
        "penalty_factor": float(best["penalty_factor"]),
        "beats_mean": bool(best["beats_mean"]),
        "difference_vs_joint_mean_b1": b1 - joint_mean_b1,
        "difference_vs_joint_mean_b2": mean_b2 - joint_mean_b2,
    }])

    return {
        "hist": hist,
        "dist": dist,
        "all_b2": all_b2,
        "best": best,
        "summary": summary,
        "mean_b2": mean_b2,
        "rounded_mean_b2": rounded_mean_b2,
    }


def render_fixed_b1_playground(
    b1=795,
    eta=0.15,
    temperature=1.0,
    n_iter=1000,
    tradable=TRADABLE,
):
    """Render all playground outputs for one manually chosen b1."""
    result = fixed_b1_playground_result(
        b1=b1,
        n_iter=n_iter,
        eta=eta,
        temperature=temperature,
        tradable=tradable,
    )
    hist = result["hist"]
    dist = result["dist"]
    all_b2 = result["all_b2"]
    best = result["best"]
    summary = result["summary"]
    mean_b2 = result["mean_b2"]
    rounded_mean_b2 = result["rounded_mean_b2"]

    display(summary.round(4))

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=hist["iter"],
        y=hist["mean_b2"],
        mode="lines",
        name="mean_b2",
        hovertemplate="iter=%{x}<br>mean_b2=%{y:.4f}<extra></extra>",
    ))
    fig.add_trace(go.Scatter(
        x=hist["iter"],
        y=hist["best_response_b2"],
        mode="lines",
        name="best-response b2 during iteration",
        line=dict(dash="dash"),
        hovertemplate="iter=%{x}<br>best_response_b2=%{y:.0f}<extra></extra>",
    ))
    fig.add_hline(
        y=rounded_mean_b2,
        line_dash="dot",
        annotation_text=f"rounded mean_b2={rounded_mean_b2}",
    )
    clean_fig(
        fig,
        f"Playground convergence for fixed b1={b1}",
        "Iteration",
        "Bid 2 level",
    )
    fig.show()

    fig = px.bar(
        dist.sort_values("b2"),
        x="b2",
        y="prob",
        hover_data={"b2": ":.0f", "prob": ":.8f"},
    )
    fig.add_vline(
        x=mean_b2,
        line_dash="dash",
        annotation_text=f"mean_b2={mean_b2:.2f}",
        annotation_position="top left",
    )
    clean_fig(
        fig,
        f"Final b2 distribution when b1 is fixed at {b1}",
        "b2",
        "Probability weight",
    )
    fig.show()

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=all_b2["b2"],
        y=all_b2["total_pnl"],
        mode="lines+markers",
        name="total PnL per counterparty",
        customdata=all_b2[["total_pnl", "first_pnl", "second_pnl", "penalty_factor", "beats_mean"]].assign(
            final_pnl_mean=all_b2.apply(
                lambda row: summarize_final_pnl(
                    b1,
                    row["b2"],
                    mean_b2,
                    tradable=tradable,
                    n_sims=1000,
                    seed=6000 + int(b1) * 10 + int(row["b2"]),
                )["final_pnl_mean"],
                axis=1,
            ),
        )[["total_pnl", "final_pnl_mean", "first_pnl", "second_pnl", "penalty_factor", "beats_mean"]],
        hovertemplate=(
            "b2=%{x:.0f}<br>total_pnl_per_counterparty=%{customdata[0]:.4f}<br>"
            "final_pnl_mean=%{customdata[1]:.2f}<br>"
            "first_pnl=%{customdata[2]:.4f}<br>second_pnl=%{customdata[3]:.4f}<br>"
            "penalty=%{customdata[4]:.4f}<br>beats_mean=%{customdata[5]}<extra></extra>"
        ),
    ))
    fig.add_vline(
        x=best["b2"],
        line_dash="dash",
        annotation_text=f"best b2={int(best['b2'])}",
        annotation_position="top left",
    )
    fig.add_vline(
        x=mean_b2,
        line_dash="dot",
        annotation_text=f"mean_b2={mean_b2:.2f}",
        annotation_position="bottom right",
    )
    clean_fig(
        fig,
        f"Best-response b2 curve for fixed b1={b1}, mean_b2≈{mean_b2:.2f}, tradable={tradable}",
        "Candidate b2",
        "Expected PnL per counterparty",
    )
    fig.show()

    show_takeaway(
        "Move the b1 slider and watch whether total_pnl rises or falls, whether the resulting mean_b2 jumps, "
        "and whether the best-response b2 remains stable. Large jumps mean the final decision should rely more on section 8 robustness."
    )

    return result


In [45]:
# Fallback: edit these values and rerun the cell if ipywidgets is unavailable.
PLAYGROUND_B1 = 795
PLAYGROUND_TRADABLE = TRADABLE

try:
    import ipywidgets as widgets
    from IPython.display import clear_output

    b1_slider = widgets.IntSlider(value=PLAYGROUND_B1, min=730, max=860, step=STEP, description="b1")
    tradable_slider = widgets.IntSlider(value=PLAYGROUND_TRADABLE, min=50, max=5000, step=50, description="tradable")
    eta_slider = widgets.FloatSlider(value=0.15, min=0.02, max=0.30, step=0.01, readout_format=".2f", description="eta")
    temperature_slider = widgets.FloatSlider(value=1.0, min=0.5, max=3.0, step=0.1, readout_format=".1f", description="temp")
    n_iter_slider = widgets.IntSlider(value=1000, min=200, max=3000, step=100, description="iters")

    out = widgets.Output()

    def _update_playground(*_):
        with out:
            clear_output(wait=True)
            render_fixed_b1_playground(
                b1=b1_slider.value,
                eta=eta_slider.value,
                temperature=temperature_slider.value,
                n_iter=n_iter_slider.value,
                tradable=tradable_slider.value,
            )

    for widget in (b1_slider, tradable_slider, eta_slider, temperature_slider, n_iter_slider):
        widget.observe(_update_playground, names="value")

    display(widgets.VBox([
        widgets.HTML("<b>Manual b1 playground</b> — move b1 / tradable and rerun the fixed-b1 mean field."),
        widgets.HBox([b1_slider, tradable_slider]),
        widgets.HBox([eta_slider, temperature_slider, n_iter_slider]),
        out,
    ]))
    _update_playground()

except Exception as exc:
    display(Markdown(
        "`ipywidgets` is unavailable in this kernel, so the interactive slider cannot render. "
        "Edit `PLAYGROUND_B1` / `PLAYGROUND_TRADABLE` above and rerun this cell instead."
    ))
    print(f"ipywidgets error: {type(exc).__name__}: {exc}")
    render_fixed_b1_playground(b1=PLAYGROUND_B1, tradable=PLAYGROUND_TRADABLE)
